# HW3 สรุปลัพธ์การคำนวณ

ปัญหา: หาเส้นทางจาก **Arad** ไปยัง **Vaslui**

1. Breadth First Tree Search
2. Depth First Tree Search
3. Breadth First Search
4. Depth First Graph Search
5. Best First Graph Search
6. Uniform Cost Search
7. Depth Limited Search
8. Iterative Deepening Search
9. Greedy Best First Search
10. A*-Search
11. Recursive Best First Search

## Setup

In [1]:
import heapq
import math
import sys

infinity = float('inf')

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = 0 if parent is None else parent.depth + 1

    def __lt__(self, other):
        return self.state < other.state

    def __eq__(self, other):
        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):
        return hash(self.state)

    def expand(self, problem):
        return [self.child_node(problem, a) for a in problem.actions(self.state)]

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        cost = problem.path_cost(self.path_cost, self.state, action, next_state)
        return Node(next_state, self, action, cost)

    def path(self):
        node, path_back = self, []
        while node:
            path_back.append(node)
            node = node.parent
        return list(reversed(path_back))

In [2]:
class Problem:
    def __init__(self, initial, goal=None):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def goal_test(self, state):
        return state == self.goal

    def path_cost(self, c, state1, action, state2):
        return c + 1

    def h(self, node):
        return 0

In [ ]:
# กราฟแผนที่โรมาเนีย 
graph = dict(
    Arad=dict(Zerind=75, Sibiu=140, Timisoara=118),
    Bucharest=dict(Urziceni=85, Pitesti=101, Giurgiu=90, Fagaras=211),
    Craiova=dict(Drobeta=120, Rimnicu=146, Pitesti=138),
    Drobeta=dict(Mehadia=75),
    Eforie=dict(Hirsova=86),
    Fagaras=dict(Sibiu=99),
    Hirsova=dict(Urziceni=98),
    Iasi=dict(Vaslui=92, Neamt=87),
    Lugoj=dict(Timisoara=111, Mehadia=70),
    Oradea=dict(Zerind=71, Sibiu=151),
    Pitesti=dict(Rimnicu=97),
    Rimnicu=dict(Sibiu=80),
    Urziceni=dict(Vaslui=142),
)

# ทำให้เป็นกราฟไม่มีทิศทาง (เดินไป-กลับได้)
for a in list(graph.keys()):
    for b, d in graph[a].items():
        graph.setdefault(b, {})[a] = d

# พิกัดเมือง (x, y) ใช้คำนวณ heuristic (ระยะทางเส้นตรงไปยัง goal)
locations = dict(
    Arad=(91, 492), Bucharest=(400, 327), Craiova=(253, 288), Drobeta=(165, 299),
    Eforie=(562, 293), Fagaras=(305, 449), Giurgiu=(375, 270), Hirsova=(534, 350),
    Iasi=(473, 506), Lugoj=(165, 379), Mehadia=(168, 339), Neamt=(406, 537),
    Oradea=(131, 571), Pitesti=(320, 368), Rimnicu=(233, 410), Sibiu=(207, 457),
    Timisoara=(94, 410), Urziceni=(456, 350), Vaslui=(509, 444), Zerind=(108, 531))


class GraphProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        return list(self.graph.get(state, {}).keys())

    def result(self, state, action):
        return action

    def path_cost(self, cost_so_far, A, action, B):
        return cost_so_far + self.graph[A][B]

    def h(self, node):
        x1, y1 = locations[node.state]
        x2, y2 = locations[self.goal]
        return math.hypot(x1 - x2, y1 - y2)


def show(node):
    if node is None or node == 'cutoff':
        print("หาไม่พบ")
        return
    print(" -> ".join(n.state for n in node.path()), " | cost =", node.path_cost)


problem = GraphProblem('Arad', 'Vaslui', graph)

## 1. Breadth First Tree Search

In [4]:
def bfts(problem):
    frontier = [Node(problem.initial)]
    while frontier:
        node = frontier.pop(0)
        if problem.goal_test(node.state):
            return node
        frontier += node.expand(problem)
    return None

show(bfts(problem))

Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui  | cost = 677


## 2. Depth First Tree Search



In [5]:
def dfts(problem):
    frontier = [Node(problem.initial)]
    while frontier:
        node = frontier.pop()
        if problem.goal_test(node.state):
            return node
        visited_in_path = {n.state for n in node.path()}
        frontier += [c for c in node.expand(problem) if c.state not in visited_in_path]
    return None

show(dfts(problem))

Arad -> Timisoara -> Lugoj -> Mehadia -> Drobeta -> Craiova -> Pitesti -> Bucharest -> Urziceni -> Vaslui  | cost = 960


## 3. Breadth First Search

In [6]:
def bfs(problem):
    node = Node(problem.initial)
    if problem.goal_test(node.state):
        return node
    frontier = [node]
    frontier_states = {node.state}
    explored = set()
    while frontier:
        node = frontier.pop(0)
        frontier_states.discard(node.state)
        explored.add(node.state)
        for child in node.expand(problem):
            if child.state not in explored and child.state not in frontier_states:
                if problem.goal_test(child.state):
                    return child
                frontier.append(child)
                frontier_states.add(child.state)
    return None

show(bfs(problem))

Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui  | cost = 677


## 4. Depth First Graph Search

In [7]:
def dfgs(problem):
    frontier = [Node(problem.initial)]
    explored = set()
    while frontier:
        node = frontier.pop()
        if problem.goal_test(node.state):
            return node
        explored.add(node.state)
        frontier_states = [n.state for n in frontier]
        frontier += [c for c in node.expand(problem)
                     if c.state not in explored and c.state not in frontier_states]
    return None

show(dfgs(problem))

Arad -> Timisoara -> Lugoj -> Mehadia -> Drobeta -> Craiova -> Pitesti -> Bucharest -> Urziceni -> Vaslui  | cost = 960


## 5. Best First Graph Search


In [ ]:
def best_first_graph_search(problem, f):
    counter = 0
    node = Node(problem.initial)
    frontier = [(f(node), counter, node)]
    best_f = {node.state: f(node)}
    explored = set()
    while frontier:
        _, _, node = heapq.heappop(frontier)
        if problem.goal_test(node.state):
            return node
        explored.add(node.state)
        for child in node.expand(problem):
            fc = f(child)
            if child.state not in explored and (child.state not in best_f or fc < best_f[child.state]):
                counter += 1
                heapq.heappush(frontier, (fc, counter, child))
                best_f[child.state] = fc
    return None

show(best_first_graph_search(problem, lambda n: problem.h(n)))

Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui  | cost = 677


## 6. Uniform Cost Search



In [9]:
def ucs(problem):
    return best_first_graph_search(problem, lambda n: n.path_cost)

show(ucs(problem))

Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucharest -> Urziceni -> Vaslui  | cost = 645


## 7. Depth Limited Search


In [10]:
def dls(problem, limit=10):
    def recurse(node, limit):
        if problem.goal_test(node.state):
            return node
        if limit == 0:
            return 'cutoff'
        cutoff = False
        for child in node.expand(problem):
            result = recurse(child, limit - 1)
            if result == 'cutoff':
                cutoff = True
            elif result is not None:
                return result
        return 'cutoff' if cutoff else None

    return recurse(Node(problem.initial), limit)

show(dls(problem, limit=10))

Arad -> Zerind -> Arad -> Zerind -> Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui  | cost = 977


## 8. Iterative Deepening Search



In [11]:
def ids(problem):
    depth = 0
    while True:
        result = dls(problem, depth)
        if result != 'cutoff':
            return result
        depth += 1

show(ids(problem))

Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui  | cost = 677


## 9. Greedy Best First Search



In [12]:
def greedy(problem):
    return best_first_graph_search(problem, lambda n: problem.h(n))

show(greedy(problem))

Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui  | cost = 677


## 10. A*-Search



In [13]:
def astar(problem):
    return best_first_graph_search(problem, lambda n: n.path_cost + problem.h(n))

show(astar(problem))

Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucharest -> Urziceni -> Vaslui  | cost = 645


## 11. Recursive Best First Search

In [14]:
def recursive_best_first_search(problem):
    def rbfs(node, f_limit):
        if problem.goal_test(node.state):
            return node, 0
        successors = node.expand(problem)
        if not successors:
            return None, math.inf
        for s in successors:
            s.f = max(s.path_cost + problem.h(s), node.f)
        while True:
            successors.sort(key=lambda n: n.f)
            best = successors[0]
            if best.f > f_limit:
                return None, best.f
            alt = successors[1].f if len(successors) > 1 else math.inf
            result, best.f = rbfs(best, min(f_limit, alt))
            if result is not None:
                return result, best.f

    root = Node(problem.initial)
    root.f = problem.h(root)
    result, _ = rbfs(root, math.inf)
    return result

show(recursive_best_first_search(problem))

Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucharest -> Urziceni -> Vaslui  | cost = 645
